# Create envelopes & derivatives

In [1]:
import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import resample_poly

def downsample_f0(f0, orig_sr, target_sr):
    # 1. Create binary voiced mask
    is_voiced = ~np.isnan(f0) & (f0 > 0)
    
    # 2. Interpolate over unvoiced/NaN gaps
    x = np.arange(len(f0))
    valid_x = x[is_voiced]
    valid_f0 = f0[is_voiced]
    
    if len(valid_x) == 0:
        return np.full(int(len(f0) * target_sr / orig_sr), np.nan)
        
    interp_func = interp1d(valid_x, valid_f0, kind='linear', bounds_error=False, fill_value="extrapolate")
    f0_continuous = interp_func(x)
    
    # 3. Downsample continuous f0 (handles anti-aliasing low-pass filtering)
    up, down = target_sr, orig_sr
    f0_downsampled = resample_poly(f0_continuous, up, down)
    
    # 4. Downsample mask using nearest-neighbor
    new_len = len(f0_downsampled)
    mask_interp = interp1d(x, is_voiced.astype(float), kind='nearest', bounds_error=False, fill_value=0)
    new_x = np.linspace(0, len(f0) - 1, new_len)
    mask_downsampled = mask_interp(new_x) > 0.5
    
    # 5. Re-mask unvoiced regions
    f0_downsampled[~mask_downsampled] = np.nan  # or 0
    return f0_downsampled



In [2]:
##Envelopes 
from scipy.signal import butter, filtfilt, find_peaks, resample
import librosa 
import naplib
import matplotlib.pyplot as plt
import logging
import numpy as np
import pandas as pd
import scipy
import pickle
from sklearn import preprocessing

min_max_scaler = preprocessing.MinMaxScaler()
logging.getLogger("naplib").setLevel(logging.FATAL)

sub='sub05'

#Load behavioral log file
path_sub = "/data/pt_02917/data/{s}/main_exp/{s}_mainexp.csv".format(s = sub)
df = pd.read_csv(path_sub, sep = ";")

target_path = "../behavior_stimuli/targets_final_norm/"
distractor_path = "../behavior_stimuli/distractors_new_clean/"

envelopes_target = []
rates_target = []
target_f0 = []

envelopes_dis = []
rates_dis = []
dis_f0 = []
stimnum = []

from scipy.stats import zscore
for i,r in df.iterrows():
    num = int(r['target'][0:3])
    print(num)
    stimnum.append(num)

    target, sr = librosa.load(target_path + r['target'], sr = 44100)
    distractor, sr = librosa.load(distractor_path + r['distractor'], sr = 44100)
    

   

    #####Target    
    ###Calculate the spectrogram 
    spec= naplib.features.auditory_spectrogram(target, sr)

    # Extract the envelope by taking the mean over frequencies
    env = np.mean(spec, axis=1)
    env = scipy.signal.resample(env, round(len(env)/125*128))    
    
    env = min_max_scaler.fit_transform(env.reshape(-1,1))
    env = env.reshape(env.shape[0])
    
    # zero pad the target
    zer = np.array([0]*64)
    y1_sil = np.concatenate((zer,env))

    ###Same for the distractor
    specdis= naplib.features.auditory_spectrogram(distractor, sr)

    # Extract the envelope by taking the mean over frequencies
    envdis = np.mean(specdis, axis=1)
    envdis = scipy.signal.resample(envdis, round(len(envdis)/125*128))    
  
    
    zer2 = np.array([0]*(len(envdis)-len(y1_sil)))
    envpad = np.concatenate((y1_sil, zer2))

    
    # Temporal derivative 
    rate = np.maximum(np.diff(envpad, prepend=envpad[0]), 0)
    rate = scipy.signal.resample(rate, len(envpad))

    
    
    #F0
    f0t,v,vp= librosa.pyin(target,sr=sr,fmin=75, fmax=175, hop_length=441)
    f0t= downsample_f0(f0t, 100, 128)
    f0t_p = np.nan_to_num(f0t, nan = np.nanmean(f0t))
   

    zer = np.array([np.mean(f0t_p)]*64)
    f0_z1 = np.concatenate((zer,f0t_p))
    zer2 = np.array([np.mean(f0t_p)]*(len(envdis)-len(f0_z1)))
    
    f0t_p = np.concatenate((f0_z1, zer2))

    env_target = min_max_scaler.fit_transform(envpad.reshape(-1,1))
    rate_target = min_max_scaler.fit_transform(rate.reshape(-1,1))

    



 
    # Temporal derivative 
    rate = np.maximum(np.diff(envdis, prepend=envdis[0]), 0)
    rate = scipy.signal.resample(rate, len(envdis))

    ##F0 
    f0d,v,vp = librosa.pyin(distractor,sr=sr,fmin=120, fmax=300, hop_length=441)
    f0d= downsample_f0(f0d, 100, 128)
    f0d_p= np.nan_to_num(f0d, nan = np.nanmean(f0d))  
    f0d_p = f0d_p[:len(envdis)] 

    zer2 = np.array([np.mean(f0d_p)]*(len(envdis)-len(f0d_p)))
    f0d_p = np.concatenate((f0d_p, zer2))

    envdis = min_max_scaler.fit_transform(envdis.reshape(-1,1))
    rate = min_max_scaler.fit_transform(rate.reshape(-1,1))


    zer = np.array([0]*64)
    env_target = env_target.reshape(env_target.shape[0])
    rate_target = rate_target.reshape(env_target.shape[0])
    
  
    
    rates_target.append(rate_target)
    envelopes_target.append(env_target)
    target_f0.append(f0t_p)

    
    rates_dis.append(rate)
    envelopes_dis.append(envdis)
    dis_f0.append(f0d_p)







        

with open("./trf_input/envelope_target.pickle".format(sub = sub), "wb") as output_file:
    pickle.dump(envelopes_target, output_file)

with open("./trf_input/envelope_dis.pickle".format(sub = sub), "wb") as output_file:
    pickle.dump(envelopes_dis, output_file)


with open("./trf_input/rate_target.pickle".format(sub = sub), "wb") as output_file:
    pickle.dump(rates_target, output_file)

with open("./trf_input/rate_dis.pickle".format(sub = sub), "wb") as output_file:
    pickle.dump(rates_dis, output_file)


with open("./trf_input/stimnum_env.pickle".format(sub = sub), "wb") as output_file:
    pickle.dump(stimnum, output_file)



   
##Standardize F0 
all_f0_t = np.concatenate(target_f0)
all_f0_d = np.concatenate(dis_f0)


mean_f0t = np.mean(all_f0_t)
std_f0t = np.std(all_f0_t)

mean_f0d = np.mean(all_f0_d)
std_f0d = np.std(all_f0_d)

f0_target_stan = []
f0_dis_stan = []
for i,f in enumerate(target_f0):
    target_f0_stan = (f-mean_f0t)/std_f0t
    dis_f0_stan = (dis_f0[i]-mean_f0d)/std_f0d
    f0_target_stan.append(target_f0_stan)
    f0_dis_stan.append(dis_f0_stan)




with open("./trf_input/f0_target.pickle", "wb") as output_file:
    pickle.dump(f0_target_stan, output_file)

with open("./trf_input/f0_dis.pickle", "wb") as output_file:
    pickle.dump(f0_dis_stan, output_file)
        

144
178
112
89
58
92
20
172
55
167
149
223
157
113
206
243
176
191
201
69
119
260
135
78
204
97
25
52
200
256
187
239
241
86
207
108
222
91
215
12
132
137
136
220
143
232
62
95
229
161
71
5
111
212
43
192
214
80
6
88
16
175
194
109
170
166
196
106
219
125
217
128
169
140
54
131
38
193
85
77
142
13
33
164
126
197
123
27
205
41
115
247
110
257
46
59
93
8
163
148
160
174
74
147
255
23
173
51
216
186
57
96
35
26
75
45
258
104
225
183
42
218
158
122
107
105
117
133
50
249
152
151
134
162
29
87
83
198
251
11
231
129
171
14
221
246
226
7
61
9
156
188
130
118
28
72
10
145
31
63
17
235
253
114
21
22
81
195
48
19
168
224
44
155
120
238
181
99
138
121
213
189
242
47
3
53
177
141
37
165
227
98
56
182
234
70
18
32
82
209
228
102
101
248
179
73
153
180
252
245
254
230
244
146
233
65
60
64
150
76
49
67
1
36
240
15
210
250
100
236
154
211
34
139
190
259
30
103
68
4
